# DisorderNet GPU: ESM-2 650M + LoRA on Colab Pro

**Goal:** AUC-ROC > 0.88 on DisProt (5-fold protein-grouped CV)

| Component | Detail |
|---|---|
| Hardware | Colab Pro GPU — A100 (40/80 GB), L4, or T4 (auto-tuned batch size) |
| Backbone | ESM-2 650M (33 layers, 1280-dim) |
| Fine-tuning | LoRA rank 8 on Q/V projections (last 8 layers) |
| Head | 1D CNN + residual skip |
| Data | DisProt (disorder-term filtered annotations) |

### Before you run
1. **Runtime → Change runtime type → GPU** (A100 or L4 recommended)
2. **Runtime → Change runtime type → High-RAM** if available
3. Run all cells top-to-bottom (~12–18 h for full 5-fold CV on A100)

Live progress: tqdm bars per epoch + per-epoch metric lines. Checkpoints saved under `checkpoints/`.

In [ ]:
# ── CELL 1 │ Clone repo (Colab only downloads the notebook by default) ──
import os, sys, subprocess

REPO_URL = "https://github.com/Tommaso-R-Marena/DisorderNet.git"
REPO_DIR = "DisorderNet"

# Already inside the cloned repo (e.g. opened from GitHub file browser)
if os.path.isfile("colab/disordernet_gpu.py"):
    if "." not in sys.path:
        sys.path.insert(0, ".")
    print("✅ Running from repository root.")
elif os.path.isdir(REPO_DIR):
    sys.path.insert(0, REPO_DIR)
    print(f"✅ Repository already at '{REPO_DIR}/'")
else:
    print("Cloning DisorderNet repository...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    sys.path.insert(0, REPO_DIR)
    print("✅ Repository cloned.")

In [ ]:
# ── CELL 2 │ Environment setup ─────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="torch")

from colab.disordernet_gpu import (
    TrainConfig,
    install_dependencies,
    setup_environment,
    fetch_disprot,
    process_disprot,
    print_dataset_summary,
    load_esm_model,
    DisorderNetGPU,
    DisProtDataset,
    disprot_collate,
    run_cross_validation,
    _in_colab,
)

install_dependencies(quiet=True)

# ── Hyperparameters (edit as needed) ──────────────────────────────────
cfg = TrainConfig(
    batch_size=4,          # auto-overridden for your GPU VRAM
    accum_steps=4,
    num_epochs=15,
    patience=4,
    n_folds=5,
    use_gradient_checkpointing=True,
    deterministic=False,   # False = TF32 + cudnn.benchmark (faster)
)

cfg = setup_environment(cfg)
DEVICE = cfg.device
print(f"\nColab: {_in_colab()}  |  Seed: {cfg.seed}")

In [ ]:
# ── CELL 3 │ Optional: mount Google Drive for checkpoint persistence ───
import os

MOUNT_DRIVE = False   # ← set True to save checkpoints across Colab sessions
DRIVE_SUBDIR = "DisorderNet"

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_ckpt = f"/content/drive/MyDrive/{DRIVE_SUBDIR}/checkpoints"
    os.makedirs(drive_ckpt, exist_ok=True)
    cfg.checkpoint_dir = drive_ckpt
    print(f"Checkpoints → {cfg.checkpoint_dir}")
else:
    print("Google Drive not mounted. Checkpoints saved locally in ./checkpoints/")

In [ ]:
# ── CELL 4 │ Download & process DisProt ────────────────────────────────
import json

entries = fetch_disprot(cache_path=cfg.data_cache)
proteins, skipped = process_disprot(entries, cfg)
total_res, total_dis, frac_dis = print_dataset_summary(proteins, skipped)

assert len(proteins) >= 500, f"Expected ≥500 proteins, got {len(proteins)}"
print("\n✅ Data validation passed.")
print("   (Only disorder-related DisProt term_names are labeled positive.")

In [ ]:
# ── CELL 5 │ Load ESM-2 650M ───────────────────────────────────────────
import torch

model_esm, alphabet, batch_converter = load_esm_model(DEVICE)

mem_gb = torch.cuda.memory_allocated() / 1e9
print(f"  VRAM used: {mem_gb:.1f} GB / {cfg.vram_gb:.1f} GB")

# Quick forward-pass sanity check
_, _, test_tokens = batch_converter([("test", "ACDEFGHIKLMNPQRSTVWY" * 3)])
with torch.inference_mode():
    out = model_esm(test_tokens.to(DEVICE), repr_layers=[33], return_contacts=False)
print(f"  Embedding shape: {out['representations'][33].shape}")
del test_tokens, out
torch.cuda.empty_cache()
print("✅ ESM-2 verified.")

In [ ]:
# ── CELL 6 │ Build model + smoke-test DataLoader ───────────────────────
from torch.utils.data import DataLoader

print("Building DisorderNetGPU...")
model = DisorderNetGPU(model_esm, cfg).to(DEVICE)
print(f"  VRAM after build: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

test_ds = DisProtDataset(proteins[:8], batch_converter)
test_dl = DataLoader(test_ds, batch_size=2, collate_fn=disprot_collate)
tok, lab, msk, ids = next(iter(test_dl))
print(f"  Batch: tokens={tok.shape}  labels={lab.shape}  disorder_frac={lab[msk].mean():.3f}")
del test_ds, test_dl, model
torch.cuda.empty_cache()
print("✅ Model + DataLoader verified.")

In [ ]:
# ── CELL 7 │ 5-Fold Cross-Validation ───────────────────────────────────
# Resume support: if Colab disconnected, set RESUME_FROM_FOLD to the next fold (0-based)
RESUME_FROM_FOLD = 0

from IPython.display import display, HTML, clear_output
import pandas as pd

live_rows = []

def _on_epoch(row):
    live_rows.append(row)
    if len(live_rows) % 1 == 0:
        clear_output(wait=True)
        display(HTML("<b>Latest training metrics</b>"))
        display(pd.DataFrame(live_rows[-10:]))

fold_results, cv_summary = run_cross_validation(
    proteins=proteins,
    esm_backbone=model_esm,
    batch_converter=batch_converter,
    cfg=cfg,
    resume_from_fold=RESUME_FROM_FOLD,
    on_epoch_end=_on_epoch,
)

pooled_auc = cv_summary["pooled_auc"]
pooled_ap  = cv_summary["pooled_ap"]
fold_aucs  = cv_summary["fold_aucs"]
total_cv_h = cv_summary["total_cv_hours"]

all_probs  = __import__("numpy").concatenate([r["val_probs"] for r in fold_results])
all_labels = __import__("numpy").concatenate([r["val_labels"] for r in fold_results])

with open("cv_results.json", "w") as f:
    json.dump(cv_summary, f, indent=2)
print("\nSaved cv_results.json")

if pooled_auc >= 0.88:
    print(f"\n🎉 TARGET AUC ≥ 0.88 ACHIEVED! ({pooled_auc:.4f})")
elif pooled_auc >= 0.83:
    print(f"\n✅ Beats DisorderNet v6 baseline (0.831). AUC = {pooled_auc:.4f}")

In [ ]:
# ── CELL 8 │ Benchmark comparison table ────────────────────────────────
from colab.colab_figures import optimal_threshold
from sklearn.metrics import f1_score, matthews_corrcoef

our_auc = pooled_auc
our_ap  = pooled_ap
opt_thresh, _ = optimal_threshold(all_labels, all_probs)
preds_opt = (all_probs >= opt_thresh).astype(int)
our_f1  = f1_score(all_labels.astype(int), preds_opt)
our_mcc = matthews_corrcoef(all_labels.astype(int), preds_opt)

BENCHMARKS = [
    ("AF3-pLDDT (CAID3)", 0.747, None, "CAID3"),
    ("DisorderNet v6 (ESM 8M+GBDT)", 0.831, 0.537, "DisProt"),
    ("ESM2_650M-LoRA", 0.880, 0.721, "CAID1"),
    ("ESMDisPred (CAID3 SOTA)", 0.895, 0.778, "CAID3"),
    ("DisorderNet GPU (OURS)", our_auc, our_ap, "DisProt"),
]
BENCHMARKS.sort(key=lambda x: x[1])

print(f"\n{'═'*64}")
print(f" {'Method':<38} {'AUC':>7} {'AP':>7}  Source")
print(f"{'─'*64}")
for method, auc, ap, source in BENCHMARKS:
    ap_s = f"{ap:.3f}" if ap else "  N/A"
    mark = " ◀" if "OURS" in method else ""
    print(f"  {method:<38} {auc:>7.3f} {ap_s:>7}  {source}{mark}")
print(f"{'═'*64}")
print(f"  F1={our_f1:.4f}  MCC={our_mcc:.4f}  opt_thresh={opt_thresh:.3f}")
print(f"  vs AF3: +{our_auc - 0.747:.4f}  |  vs v6: +{our_auc - 0.831:.4f}")

In [ ]:
# ── CELL 9 │ Figures ───────────────────────────────────────────────────
from colab.colab_figures import generate_all_figures

fig_metrics = generate_all_figures(
    fold_results, all_labels, all_probs, our_auc, our_ap
)
opt_thresh = fig_metrics["opt_thresh"]
our_f1     = fig_metrics["f1"]
our_mcc    = fig_metrics["mcc"]
print("✅ All figures saved.")

In [ ]:
# ── CELL 10 │ Save results & best checkpoint ───────────────────────────
import shutil
import numpy as np
from datetime import datetime

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
best_fold_idx = int(np.argmax([r["best_auc"] for r in fold_results]))
best_fold     = fold_results[best_fold_idx]
best_ckpt_dst = f"disordernet_gpu_best_{RUN_TIMESTAMP}.pt"

if best_fold.get("ckpt_path") and os.path.exists(best_fold["ckpt_path"]):
    shutil.copy(best_fold["ckpt_path"], best_ckpt_dst)
    print(f"Best checkpoint: {best_ckpt_dst}  (fold {best_fold['fold']}, AUC={best_fold['best_auc']:.4f})")

full_results = {
    "run_timestamp": RUN_TIMESTAMP,
    "config": cv_summary["config"],
    "dataset": {"n_proteins": len(proteins), "n_residues": int(total_res), "frac_disorder": float(frac_dis)},
    "pooled": {"auc": float(our_auc), "ap": float(our_ap), "f1": float(our_f1),
               "mcc": float(our_mcc), "opt_threshold": float(opt_thresh)},
    "per_fold": [{"fold": r["fold"], "best_auc": r["best_auc"], "best_ap": r["best_ap"],
                  "train_time_h": r["total_time"] / 3600, "history": r["history"]} for r in fold_results],
    "best_checkpoint": best_ckpt_dst,
}
results_path = f"disordernet_gpu_results_{RUN_TIMESTAMP}.json"
with open(results_path, "w") as f:
    json.dump(full_results, f, indent=2)

print(f"\n{'═'*60}")
print(" DISORDERNET GPU — FINAL RESULTS")
print(f"{'═'*60}")
print(f"  AUC-ROC  : {our_auc:.4f}")
print(f"  Avg Prec : {our_ap:.4f}")
print(f"  F1       : {our_f1:.4f}")
print(f"  Total CV : {total_cv_h:.2f} h")
print(f"  Saved    : {results_path}")
print(f"{'═'*60}")